# Papers with Code evaluation (Hugging Face)

Compares the extraction notebook **Combinations** sheet to [`pwc-archive/evaluation-tables`](https://huggingface.co/datasets/pwc-archive/evaluation-tables).

## Requirements

- Run the extraction notebook first so `gliner2_lightonocr_combinations_{raw|filtered}.xlsx` exists (typically under `table_extraction/kge_corpus/`).
- In **this Jupyter kernel**: `pip install datasets openpyxl`

## What this notebook does

- **Configuration** cell: paths, optional `nbconvert` extraction, `TASK_FILTER`, output Excel.
- **Optional:** `ENRICH_DATASET_JSON` updates only `metrics` in `data/dataset_con_rutas_metrics_xml.json` from `*lightonocr*.json`; console-only overlap vs PwC by repo URL (streaming). PwC metrics are **not** written into that JSON.
- **Evaluation:** Combinations vs PwC (normalized metric names). With `EVAL_COMBINATIONS_USE_JSON_IDS=True`, paper keys are `arxiv:` / `slug:` / `title:` (fuzzy fallback **only** between `title:` keys).

## Output

`table_extraction/evaluation_against_pwc.xlsx`: `global_scores`, `coverage_missing`, `papers_without_metrics`.

## Troubleshooting

- **`KeyError: 'maxdepth'`** when loading the dataset: run **Upgrade packages** with `UPGRADE_HF_LIBS_IN_THIS_KERNEL=True`, **restart the kernel**, re-run; or  
  `python -m pip install -U "datasets>=3.0.0" "huggingface_hub>=0.26.0" "fsspec>=2024.10.0"`
- **Hub limits (optional):** `HF_TOKEN`, `huggingface-cli login`, or **Optional HF login** (`RUN_HF_LOGIN=True`; use the widget, not a string in the notebook).

**PwC schema / sample rows:** run the last diagnostic code cell.


In [ ]:
# Optional: bump HF stack in THIS kernel for Hub dataset loads
import subprocess
import sys

UPGRADE_HF_LIBS_IN_THIS_KERNEL = True  # Set False once stable to skip reinstalling each run

if UPGRADE_HF_LIBS_IN_THIS_KERNEL:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            "datasets>=3.0.0",
            "huggingface_hub>=0.26.0",
            "fsspec>=2024.10.0",
        ]
    )
    print("Upgraded with:", sys.executable)
    print("Restart the kernel and re-run from the first imports cell.")
    print("If the error persists after restart, in the same env: conda install -c conda-forge 'fsspec>=2024.10.0'")
else:
    print("Upgrade skipped.")

In [ ]:
from __future__ import annotations

import json
import os
import difflib
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd

## Optional Hugging Face login

Improves Hub rate limits. In the next code cell set `RUN_HF_LOGIN=True` and paste your [token](https://huggingface.co/settings/tokens) in the **widget** (do not commit tokens in notebook source).

Alternatively: `huggingface-cli login` or `export HF_TOKEN=...` before Jupyter.


In [ ]:
# Optional: Hugging Face login (better Hub rate limits)
RUN_HF_LOGIN = False  # Set True to authenticate on this machine

if RUN_HF_LOGIN:
    from huggingface_hub import login

    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:
        print("notebook_login unavailable in this frontend; falling back to login():", exc)
        login()
else:
    print(
        "HF login skipped (RUN_HF_LOGIN=False).\n"
        "HF_TOKEN or a prior CLI login is enough.\n"
        "To use the widget: set RUN_HF_LOGIN=True and re-run this cell."
    )

## Unified pipeline (extraction + evaluation)

Controlled in the **Configuration** cell: `RUN_EXTRACTION_FIRST`, `AUTO_RUN_EXTRACTION_IF_MISSING`.  
Do not edit `evaluation_table_extraction_gliner2_lightonocr.ipynb`: `nbconvert` temporarily symlinks `table_extraction/pdfs_prueba` → `kge_corpus` so outputs go to the persistent corpus; a pre-existing `pdfs_prueba/` directory is backed up and restored afterward.


In [ ]:
# Pointer only — use the **Configuration** cell below.
# RUN_EXTRACTION_FIRST / AUTO_RUN_EXTRACTION_IF_MISSING and the pdfs_prueba symlink logic live there.
# Symlink: table_extraction/pdfs_prueba -> kge_corpus (extraction notebook unchanged).
print("[pipeline] See Configuration: RUN_EXTRACTION_FIRST / AUTO_RUN_EXTRACTION_IF_MISSING.")

In [ ]:
# Configuration
import shutil
import subprocess
import sys
# Must match export mode in the extraction notebook:
#   filtered → USE_BLACKLIST True   |   raw → USE_BLACKLIST False
MODE = "filtered"
REQUIRE_PDF_FILES_CORPUS = True
AUTO_RUN_EXTRACTION_IF_MISSING = True  # if the Excel is missing, run extraction from this notebook
# True = always run extraction (nbconvert) before resolving INPUT_EXCEL
RUN_EXTRACTION_FIRST = False
# Persistent corpus (PDFs + JSON caches + Excel), synced from data/pdf_files
STABLE_CORPUS_DIR = Path("table_extraction/kge_corpus")
# data/dataset_con_rutas_metrics_xml.json: only `metrics` from GLiNER2+LightOCR JSON in the corpus.
ENRICH_DATASET_JSON = True
DATASET_JSON_BASENAME = "dataset_con_rutas_metrics_xml.json"
STREAM_PWC_FOR_REPO_METRICS = True
# True: Combinations vs PwC with arxiv/slug from dataset JSON + PwC row keys (better than title-only fuzzy)
EVAL_COMBINATIONS_USE_JSON_IDS = True


def _resolve_output_excel() -> Path:
    for p in (Path("evaluation_against_pwc.xlsx"), Path("table_extraction/evaluation_against_pwc.xlsx")):
        if p.parent == Path(".") or p.parent.is_dir():
            return p
    return Path("evaluation_against_pwc.xlsx")


def _find_repo_root_for_extraction() -> Path:
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "table_extraction/evaluation_table_extraction_gliner2_lightonocr.ipynb").exists():
            return c.resolve()
    raise FileNotFoundError("Repository root not found for extraction.")


def _ensure_kge_corpus(repo_root: Path) -> Path:
    """Populate table_extraction/kge_corpus with symlinks (or copies) from data/pdf_files.
    Idempotent: does not delete existing JSON/Excel artifacts.
    """
    source = (repo_root / "data" / "pdf_files").resolve()
    stable = (repo_root / STABLE_CORPUS_DIR).resolve()
    if not source.is_dir():
        raise FileNotFoundError(f"Source corpus not found: {source}")
    stable.mkdir(parents=True, exist_ok=True)
    for pdf in sorted(source.glob("*.pdf")):
        dest = stable / pdf.name
        if dest.exists() or dest.is_symlink():
            continue
        try:
            dest.symlink_to(pdf)
        except OSError:
            shutil.copy2(pdf, dest)
    return stable


def _execute_extraction_notebook_via_kge_corpus(repo_root: Path) -> None:
    """Extraction notebook still uses `pdfs_prueba/`; for nbconvert we symlink
    table_extraction/pdfs_prueba -> kge_corpus so caches/Excel land there without editing that .ipynb.
    """
    stable = _ensure_kge_corpus(repo_root)
    alias = repo_root / "table_extraction" / "pdfs_prueba"
    backup = repo_root / "table_extraction" / "pdfs_prueba.__eval_unified_backup__"
    if backup.exists():
        raise RuntimeError(
            f"Stale backup folder (interrupted run?): {backup}. "
            "Restore or delete it and retry."
        )

    linked_by_us = False
    restore_dir = False
    try:
        if not (alias.is_symlink() and alias.resolve() == stable):
            if alias.is_symlink():
                alias.unlink()
            if alias.exists():
                alias.rename(backup)
                restore_dir = True
            alias.symlink_to(stable)
            linked_by_us = True

        nb = repo_root / "table_extraction" / "evaluation_table_extraction_gliner2_lightonocr.ipynb"
        print(
            "[auto-extract] Running extraction into persistent corpus (pdfs_prueba symlink):",
            stable.resolve(),
        )
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "jupyter",
                "nbconvert",
                "--to",
                "notebook",
                "--execute",
                "--inplace",
                str(nb),
            ],
            cwd=repo_root,
        )
    finally:
        if linked_by_us:
            if alias.is_symlink():
                alias.unlink()
            if restore_dir and backup.exists():
                backup.rename(alias)


def _run_extraction_over_pdf_files() -> None:
    _execute_extraction_notebook_via_kge_corpus(_find_repo_root_for_extraction())
    print("[auto-extract] Extraction finished.")


if RUN_EXTRACTION_FIRST:
    print("[unified] RUN_EXTRACTION_FIRST=True: running full extraction before resolving INPUT_EXCEL")
    _execute_extraction_notebook_via_kge_corpus(_find_repo_root_for_extraction())
    print("[unified] Forced extraction finished.")


def _resolve_input_excel(mode: str, require_pdf_files: bool = True) -> Path:
    fname = f"gliner2_lightonocr_combinations_{mode}.xlsx"
    repo_root = _find_repo_root_for_extraction()

    primary_roots = [
        (repo_root / STABLE_CORPUS_DIR),
        (repo_root / "pdf_files"),
        (repo_root / "data" / "pdf_files"),
        (repo_root / "table_extraction" / "pdf_files"),
        (repo_root / "table_extraction" / "data" / "pdf_files"),
    ]
    fallback_roots = [
        (repo_root / "pdfs_prueba"),
        (repo_root / "table_extraction" / "pdfs_prueba"),
    ]

    def _search(roots: list[Path]) -> Path | None:
        for root in roots:
            p = root / fname
            if p.exists():
                return p.resolve()
        for root in roots:
            if root.is_dir():
                found = sorted(
                    root.glob("gliner2_lightonocr_combinations_*.xlsx"),
                    key=lambda x: x.stat().st_mtime,
                    reverse=True,
                )
                if found:
                    return found[0].resolve()
        return None

    p = _search(primary_roots)
    if p is not None:
        return p

    if require_pdf_files and AUTO_RUN_EXTRACTION_IF_MISSING:
        _run_extraction_over_pdf_files()
        p = _search(primary_roots)
        if p is not None:
            return p

    if require_pdf_files:
        searched = "\n - ".join(str(x) for x in primary_roots)
        raise FileNotFoundError(
            f"Could not find {fname} under primary paths after auto-extraction. Searched:\n - {searched}"
        )

    p = _search(fallback_roots)
    if p is not None:
        return p

    raise FileNotFoundError(f"Could not find {fname} in any known path.")


INPUT_EXCEL = _resolve_input_excel(MODE, require_pdf_files=REQUIRE_PDF_FILES_CORPUS)
INPUT_SHEET = "Combinations"
OUTPUT_EXCEL = _resolve_output_excel()
TASK_FILTER = "link prediction"  # "" = all tasks in the dataset
MATCH_THRESHOLD = 0.60

print(f"INPUT_EXCEL  = {INPUT_EXCEL}")
print(f"OUTPUT_EXCEL = {OUTPUT_EXCEL}")

In [ ]:
METRIC_ALIASES = {
    "mrr": "mrr",
    "meanreciprocalrank": "mrr",
    "meanrank": "mr",
    "mr": "mr",
    "hit@1": "hits@1",
    "hits@1": "hits@1",
    "hit@3": "hits@3",
    "hits@3": "hits@3",
    "hit@10": "hits@10",
    "hits@10": "hits@10",
    "hits10": "hits@10",
    "f1": "f1",
    "f1score": "f1",
    "f1measure": "f1",
    "accuracy": "accuracy",
    "acc": "accuracy",
    "auc": "auc",
    "map": "map",
}

PAPER_COL_CANDIDATES = ["paper_title", "paper", "title", "paper_name", "paper_url", "url"]
DATASET_COL_CANDIDATES = ["dataset", "dataset_name", "eval_dataset", "benchmark"]
METRIC_COL_CANDIDATES = ["metric", "metric_name", "evaluation_metric", "metrics"]
TASK_COL_CANDIDATES = ["task", "task_name", "subtask"]

In [ ]:
def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents(s)
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_paper(text: object) -> str:
    s = normalize_text(text)
    if s.startswith("http"):
        s = s.rstrip("/")
        s = s.split("/")[-1]
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


def normalize_dataset(text: object) -> str:
    s = normalize_text(text)
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    s = s.replace("%", "").replace("(", "").replace(")", "")
    s = s.replace(" ", "").replace("-", "").replace("_", "")
    s = s.replace("hitsat", "hits@").replace("hitat", "hit@").replace("h@", "hits@")
    if s in METRIC_ALIASES:
        return METRIC_ALIASES[s]
    m = re.search(r"hits@?(\d+)", s)
    if m:
        return f"hits@{m.group(1)}"
    return s


def safe_f1(p: float, r: float) -> float:
    return 0.0 if (p + r) == 0 else (2 * p * r) / (p + r)


def first_present_column(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols:
            return cols[c.lower()]
    return None

In [ ]:
# Enrich data/dataset_con_rutas_metrics_xml.json in place (metrics field only)
# Each JSON object: metrics = names parsed from GLiNER2+LightOCR JSON in the corpus, or [].
# Stream PwC by repo URL → normalized metric names (console only; not written to JSON).
# Deduplicate: one row per normalized title (`normalize_paper`).


def _local_path_pick_score(repo_root: Path, lp: object) -> tuple[int, int, int]:
    s = "" if lp is None else str(lp).strip().replace("\\", "/")
    if not s:
        return (-2, 0, 0)
    name = Path(s).name
    variants = [
        repo_root / s,
        repo_root / s.lstrip("./"),
        repo_root / "data" / name,
        repo_root / "data" / "pdf_files" / name,
    ]
    variants.append((repo_root / STABLE_CORPUS_DIR / name).resolve())
    exists = any(v.is_file() for v in variants if hasattr(v, "is_file"))
    mtd = "model_type_pdfs" in s.lower()
    return (2 if exists else 0, 1 if mtd else 0, len(s))


def _dedupe_papers_list_by_normalized_title(repo_root: Path, papers: list) -> tuple[list, int]:
    from collections import defaultdict

    nondict_suffix = [o for o in papers if not isinstance(o, dict)]
    buckets: dict[str, list[dict]] = defaultdict(list)
    order: list[str] = []
    for o in papers:
        if not isinstance(o, dict):
            continue
        k = normalize_paper(o.get("title"))
        if not k:
            uid = f"__notitle::{id(o)}"
            buckets[uid] = [o]
            order.append(uid)
            continue
        if k not in buckets:
            order.append(k)
        buckets[k].append(o)
    out: list = []
    removed = 0
    for k in order:
        rows = buckets[k]
        if len(rows) == 1:
            out.append(dict(rows[0]))
            continue
        removed += len(rows) - 1
        best = max(
            range(len(rows)),
            key=lambda i: (_local_path_pick_score(repo_root, rows[i].get("local_path")), i),
        )
        out.append(dict(rows[best]))
    out.extend(nondict_suffix)
    return out, removed


def _hf_token_stream() -> str | None:
    t = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    return t.strip() if t and str(t).strip() else None


def _canonical_repo(url: object) -> str:
    if url is None or str(url).strip() == "":
        return ""
    u = str(url).strip().rstrip("/")
    if u.lower().endswith(".git"):
        u = u[:-4]
    return u.lower()


_RAW_METRIC_PATTERNS: list[tuple[re.Pattern, object]] = [
    (re.compile(r"(?i)hits?\s*@?\s*(\d+)"), lambda m: f"hits@{int(m.group(1))}"),
    (re.compile(r"(?i)mean\s*reciprocal\s*rank"), lambda _: "mrr"),
    (re.compile(r"(?i)\bmrr\b"), lambda _: "mrr"),
    (re.compile(r"(?i)mean\s+rank\b"), lambda _: "mr"),
    (re.compile(r"(?i)\bmr\b"), lambda _: "mr"),
    (re.compile(r"(?i)\bmap\b"), lambda _: "map"),
    (re.compile(r"(?i)\b(?:auc|roc)\b"), lambda _: "auc"),
]


def _iter_strings_deep(x):
    if isinstance(x, str):
        yield x
        return
    if isinstance(x, dict):
        for k, v in x.items():
            if isinstance(k, str):
                yield k
            yield from _iter_strings_deep(v)
        return
    if isinstance(x, (list, tuple)):
        for y in x:
            yield from _iter_strings_deep(y)


def _metrics_from_lightonocr_payload(payload: object) -> set[str]:
    found: set[str] = set()
    for txt in _iter_strings_deep(payload):
        t = " ".join(str(txt).split())
        if not t:
            continue
        for rx, canon in _RAW_METRIC_PATTERNS:
            for m in rx.finditer(t):
                nm = normalize_metric(str(canon(m)))
                if nm:
                    found.add(nm)
    return found


def _paper_title_candidates_from_payload(payload: object, path: Path) -> set[str]:
    out: set[str] = set()
    if isinstance(payload, dict):
        for key in ("paper_title", "title"):
            val = payload.get(key)
            n = normalize_paper(val)
            if n:
                out.add(n)
        docs = payload.get("documents")
        if isinstance(docs, list):
            for d in docs:
                if isinstance(d, dict):
                    n = normalize_paper(d.get("paper_title") or d.get("title"))
                    if n:
                        out.add(n)
    stem = path.stem
    stem = re.sub(r"(?i)_?gliner2_lightonocr$", "", stem)
    stem = re.sub(r"(?i)_?lightonocr$", "", stem)
    stem = stem.replace("_", " ")
    n_stem = normalize_paper(stem)
    if n_stem:
        out.add(n_stem)
    return out


def _collect_lightonocr_metrics(repo_root: Path) -> tuple[dict[str, set[str]], dict[str, list[str]]]:
    cand_dirs = [
        (repo_root / STABLE_CORPUS_DIR).resolve(),
        (repo_root / "table_extraction" / "kge_corpus").resolve(),
        (repo_root / "table_extraction" / "pdfs_prueba").resolve(),
    ]
    title_to_metrics: dict[str, set[str]] = {}
    title_to_paths: dict[str, list[str]] = {}
    seen: set[Path] = set()
    for d in cand_dirs:
        if not d.is_dir():
            continue
        for p in sorted(d.rglob("*lightonocr*.json")):
            if p in seen:
                continue
            seen.add(p)
            if "ground_truth" in {part.lower() for part in p.parts}:
                continue
            try:
                payload = json.loads(p.read_text(encoding="utf-8"))
            except Exception:
                continue
            mets = _metrics_from_lightonocr_payload(payload)
            keys = _paper_title_candidates_from_payload(payload, p)
            for k in keys:
                title_to_metrics.setdefault(k, set()).update(mets)
                title_to_paths.setdefault(k, []).append(str(p.resolve()))
    return title_to_metrics, title_to_paths


def _resolve_lightonocr_by_title_key(
    paper_key: str,
    title_to_metrics: dict[str, set[str]],
    title_to_paths: dict[str, list[str]],
) -> tuple[set[str], list[str], str]:
    if not paper_key:
        return set(), [], "missing_in_lightonocr"

    # 1) Match exacto (preferido)
    if paper_key in title_to_metrics:
        return (
            set(title_to_metrics.get(paper_key, set())),
            sorted(set(title_to_paths.get(paper_key, []))),
            "matched_lightonocr_exact",
        )

    # 2) Fallback for truncated titles in filenames
    pk_toks = paper_key.split()
    near_keys: list[str] = []
    for k in title_to_metrics.keys():
        if not k:
            continue
        # Long shared prefix (typical truncation)
        pref = (paper_key.startswith(k) or k.startswith(paper_key)) and min(len(paper_key), len(k)) >= 28
        if pref:
            near_keys.append(k)
            continue
        # Small token-set gap (<=1 token length difference)
        kk_toks = k.split()
        if abs(len(pk_toks) - len(kk_toks)) <= 1:
            overlap = len(set(pk_toks) & set(kk_toks))
            if overlap >= max(1, min(len(pk_toks), len(kk_toks)) - 1):
                near_keys.append(k)

    if not near_keys:
        return set(), [], "missing_in_lightonocr"

    mx: set[str] = set()
    srcs: set[str] = set()
    for nk in near_keys:
        mx.update(title_to_metrics.get(nk, set()))
        srcs.update(title_to_paths.get(nk, []))
    return mx, sorted(srcs), "matched_lightonocr_fuzzy"


def _hf_row_as_list(val) -> list:
    if val is None:
        return []
    if hasattr(val, "tolist"):
        try:
            val = val.tolist()
        except Exception:
            pass
    if isinstance(val, (list, tuple)):
        return list(val)
    return [val]


def _streaming_repo_to_normalized_metrics(task_substr: str) -> dict[str, set[str]]:
    from datasets import load_dataset

    out: dict[str, set[str]] = {}
    task_substr = normalize_text(task_substr).strip()

    ds = load_dataset(
        "pwc-archive/evaluation-tables",
        split="train",
        streaming=True,
        token=_hf_token_stream(),
    )

    i = 0
    for row in ds:
        i += 1
        if i % 300 == 0:
            print(f"  [streaming PwC repos] ~{i} task-rows scanned...")

        tn = normalize_text(row.get("task", ""))
        if task_substr and task_substr not in tn:
            continue

        for ds_entry in _hf_row_as_list(row.get("datasets")):
            if not isinstance(ds_entry, dict):
                continue
            sota = ds_entry.get("sota") if isinstance(ds_entry.get("sota"), dict) else None
            if not sota:
                continue
            for rr in _hf_row_as_list(sota.get("rows")):
                if not isinstance(rr, dict):
                    continue
                met_raw = rr.get("metrics") or {}
                metric_names_norm: list[str] = []
                if isinstance(met_raw, dict):
                    for k in met_raw.keys():
                        nm = normalize_metric(str(k))
                        if nm:
                            metric_names_norm.append(nm)
                if not metric_names_norm:
                    continue
                links = rr.get("code_links") or []
                for lk in _hf_row_as_list(links):
                    if not isinstance(lk, dict):
                        continue
                    u = lk.get("url")
                    rk = _canonical_repo(u)
                    if not rk:
                        continue
                    if rk not in out:
                        out[rk] = set()
                    out[rk].update(metric_names_norm)
    return out


json_metrics_xml_vs_pwc_df = pd.DataFrame()

_repo_for_json = _find_repo_root_for_extraction()
_JSON_DATASET_PATH = (_repo_for_json / "data" / DATASET_JSON_BASENAME).resolve()


if ENRICH_DATASET_JSON:
    if not _JSON_DATASET_PATH.is_file():
        raise FileNotFoundError(f"Missing {_JSON_DATASET_PATH}")
    print(f"Dataset JSON (read/write): {_JSON_DATASET_PATH}")
    with open(_JSON_DATASET_PATH, encoding="utf-8") as f:
        papers_enrich = json.load(f)

    papers_enrich, _dup_removed = _dedupe_papers_list_by_normalized_title(_repo_for_json, papers_enrich)
    if _dup_removed:
        print(
            "Deduplicated by normalized `title`: removed "
            f"{_dup_removed} entries; {sum(1 for x in papers_enrich if isinstance(x, dict))} unique objects remain."
        )

    print("Building paper → metrics map from *lightonocr*.json in the corpus...")
    title_to_metrics, title_to_paths = _collect_lightonocr_metrics(_repo_for_json)
    print(f"Papers seen in LightOCR extraction JSONs: {len(title_to_metrics)}")

    repo_map_metrics: dict[str, set[str]] = {}
    if STREAM_PWC_FOR_REPO_METRICS:
        print(f"Building repo → PwC metrics map (streaming), task filter ~ {TASK_FILTER!r}...")
        repo_map_metrics = _streaming_repo_to_normalized_metrics(TASK_FILTER)
        print(f"Repos with PwC metrics: {len(repo_map_metrics)}")

    audit_rows = []
    for obj in papers_enrich:
        if not isinstance(obj, dict):
            continue
        paper_key = normalize_paper(obj.get("title"))
        mx, srcs, mapping_state = _resolve_lightonocr_by_title_key(
            paper_key,
            title_to_metrics,
            title_to_paths,
        )
        ru = _canonical_repo(obj.get("repo_url"))
        mp = repo_map_metrics.get(ru, set()) if ru else set()
        inter = mx & mp

        obj.pop("metrics_pwc_repo", None)
        obj["metrics"] = sorted(mx)

        prec = len(inter) / len(mx) if mx else ""
        rec = len(inter) / len(mp) if mp else ""

        audit_rows.append(
            {
                "arxiv_id": obj.get("arxiv_id"),
                "title": obj.get("title"),
                "paper_norm_title": normalize_paper(obj.get("title")),
                "repo_url": obj.get("repo_url"),
                "repo_canon": ru,
                "mapping_state": mapping_state,
                "lightonocr_json_paths": " | ".join(srcs),
                "n_metrics_xml": len(mx),
                "n_metrics_pwc_repo": len(mp),
                "n_intersection": len(inter),
                "precision_xml_in_intersection_vs_xml": round(float(prec), 4) if prec != "" else "",
                "recall_xml_in_intersection_vs_pwc_repo": round(float(rec), 4) if rec != "" else "",
                "only_in_xml": ", ".join(sorted(mx - mp)),
                "only_in_pwc_repo": ", ".join(sorted(mp - mx)),
                "metrics_xml_flat": ", ".join(sorted(mx)),
                "metrics_pwc_repo_flat": ", ".join(sorted(mp)),
            }
        )

    json_metrics_xml_vs_pwc_df = pd.DataFrame(audit_rows)

    if STREAM_PWC_FOR_REPO_METRICS and not json_metrics_xml_vs_pwc_df.empty:
        d = json_metrics_xml_vs_pwc_df
        rc = pd.to_numeric(d["n_metrics_xml"], errors="coerce").fillna(0)
        pc = pd.to_numeric(d["n_metrics_pwc_repo"], errors="coerce").fillna(0)
        inter = pd.to_numeric(d["n_intersection"], errors="coerce").fillna(0)
        has_repo = (d["repo_canon"] != "").sum()
        print("\n=== LightOCR metrics vs PwC (by repo URL, streaming, TASK_FILTER) ===")
        print(f"Group metric keys from SoTA rows and canonicalize code_links[].url → repo key.")
        print(f"Rows after deduplication: {len(d)}")
        print(f"Con repo_url: {has_repo}")
        print(f"With non-empty LightOCR metrics: {int((rc > 0).sum())}")
        print(f"With non-empty PwC metrics for that repo: {int((pc > 0).sum())}")
        print(f"With non-empty metric-name intersection: {int((inter > 0).sum())}")
        mk = rc > 0
        if mk.any():
            psub = pd.to_numeric(d.loc[mk, "precision_xml_in_intersection_vs_xml"], errors="coerce").dropna()
            print(f"Mean precision where LightOCR has metrics: {round(float(psub.mean()), 4) if len(psub) else 'n/a'}")

    with open(_JSON_DATASET_PATH, "w", encoding="utf-8") as f:
        json.dump(papers_enrich, f, indent=2, ensure_ascii=False)

    dataset_keys = {normalize_paper(o.get("title")) for o in papers_enrich if isinstance(o, dict)}
    extracted_keys = set(title_to_metrics.keys())
    missing_in_extraction = sorted(k for k in dataset_keys if k and k not in extracted_keys)
    print(f"JSON updated (metrics field only per entry): {_JSON_DATASET_PATH}")
    print(f"JSON papers in audit table: {len(json_metrics_xml_vs_pwc_df)}")
    print(f"JSON papers without a mapped lightonocr JSON: {len(missing_in_extraction)}")
    print(f"JSON papers with metrics=[]: {int((json_metrics_xml_vs_pwc_df['n_metrics_xml'] == 0).sum())}")
else:
    print("ENRICH_DATASET_JSON=False: skip JSON enrichment.")

In [ ]:
def load_ours_combinations(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input Excel not found: {path}")
    df = pd.read_excel(path, sheet_name=sheet_name)
    colmap = {c.lower(): c for c in df.columns}
    missing = {"paper", "dataset", "metric"} - set(colmap.keys())
    if missing:
        raise ValueError(f"Sheet '{sheet_name}' missing columns: {missing}")
    out = pd.DataFrame({
        "paper_raw": df[colmap["paper"]],
        "dataset_raw": df[colmap["dataset"]],
        "metric_raw": df[colmap["metric"]],
    })
    out["paper_norm"] = out["paper_raw"].map(normalize_paper)
    out["dataset_norm"] = out["dataset_raw"].map(normalize_dataset)
    out["metric_norm"] = out["metric_raw"].map(normalize_metric)
    out = out[(out["paper_norm"] != "") & (out["metric_norm"] != "")]
    return out.drop_duplicates()


def _hf_dataset_token() -> str | None:
    """HF token from the environment if set.
    If None, `load_dataset` uses the Hub default (cached CLI / notebook login when present)."""
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    return tok.strip() if tok and str(tok).strip() else None

def _pwc_as_sequence(x) -> list:
    if x is None:
        return []
    if hasattr(x, "tolist"):
        try:
            x = x.tolist()
        except Exception:
            return []
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def _pwc_walk_dataset_nodes(d: dict):
    yield d
    for sd in _pwc_as_sequence(d.get("subdatasets")):
        if isinstance(sd, dict):
            yield from _pwc_walk_dataset_nodes(sd)


def _arxiv_norm_from_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    m = re.search(r"(\d{4}\.\d{4,5})(?:v\d+)?", s)
    return m.group(1) if m else ""


def _paperswithcode_slug_from_url(url: object) -> str:
    if url is None:
        return ""
    s = str(url).strip().rstrip("/")
    if "paperswithcode.com/paper/" in s:
        return s.split("paperswithcode.com/paper/", 1)[-1].split("/")[0].strip().lower()
    return ""


def _pwc_identity_from_row(mrow: dict | None, paper_raw: str) -> tuple[str, str]:
    """Stable paper key aligned with dataset JSON: arxiv > PapersWithCode slug > normalized title."""
    mr = mrow if isinstance(mrow, dict) else {}
    for k in ("arxiv_id", "arxiv", "paper_arxiv_id"):
        a = _arxiv_norm_from_text(mr.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    for ukey in ("paper_url", "url"):
        sl = _paperswithcode_slug_from_url(mr.get(ukey))
        if sl:
            return f"slug:{sl}", "slug"
    blob = " ".join(str(mr.get(x, "") or "") for x in ("paper_title", "paper", "paper_url"))
    mb = re.search(r"\b(\d{4}\.\d{5})\b", blob) or re.search(r"\b(\d{4}\.\d{4})\b", blob)
    if mb:
        return f"arxiv:{mb.group(1)}", "arxiv"
    a2 = _arxiv_norm_from_text(paper_raw)
    if a2:
        return f"arxiv:{a2}", "arxiv"
    sl2 = _paperswithcode_slug_from_url(paper_raw)
    if sl2:
        return f"slug:{sl2}", "slug"
    return f"title:{normalize_paper(paper_raw)}", "title"


def _dataset_json_identity(o: dict) -> tuple[str, str]:
    for k in ("arxiv_id", "url_abs", "url_pdf", "paper_url"):
        a = _arxiv_norm_from_text(o.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    sl = _paperswithcode_slug_from_url(o.get("paper_url"))
    if sl:
        return f"slug:{sl}", "slug"
    return f"title:{normalize_paper(o.get('title'))}", "title"


def _pwc_flatten_nested_eval_tables(df: pd.DataFrame, task_filter: str = "") -> pd.DataFrame:
    """Flatten pwc-archive/evaluation-tables nested parquet (task -> datasets[] -> sota.rows[])."""
    rows_out: list[dict[str, object]] = []
    tf_full = normalize_text(task_filter) if task_filter else ""

    for _, r in df.iterrows():
        task_name = r.get("task", "")
        if tf_full:
            if tf_full not in normalize_text(str(task_name)):
                continue

        for d in _pwc_as_sequence(r.get("datasets")):
            if not isinstance(d, dict):
                continue
            for node in _pwc_walk_dataset_nodes(d):
                ds_name = str(node.get("dataset") or "").strip()
                sota = node.get("sota") if isinstance(node.get("sota"), dict) else None
                if not sota:
                    continue
                for mrow in _pwc_as_sequence(sota.get("rows")):
                    if not isinstance(mrow, dict):
                        continue
                    paper_title = (
                        mrow.get("paper_title") or mrow.get("paper") or mrow.get("paper_url") or ""
                    )
                    paper_title = str(paper_title).strip()
                    mk, mk_kind = _pwc_identity_from_row(mrow, paper_title)
                    met = mrow.get("metrics")
                    if not isinstance(met, dict):
                        continue
                    for mn, mv in met.items():
                        if mv is None or str(mn).strip() == "":
                            continue
                        rows_out.append({
                            "paper_raw": paper_title,
                            "dataset_raw": ds_name,
                            "metric_raw": mn,
                            "task_raw": task_name,
                            "paper_match_key": mk,
                            "paper_match_kind": mk_kind,
                        })

    if not rows_out:
        raise RuntimeError(
            "No rows left after flattening pwc-archive/evaluation-tables "
            "(check TASK_FILTER or dataset revision)."
        )
    flat = pd.DataFrame(rows_out)
    flat["paper_norm"] = flat["paper_raw"].map(normalize_paper)
    flat["dataset_norm"] = flat["dataset_raw"].map(normalize_dataset)
    flat["metric_norm"] = flat["metric_raw"].map(normalize_metric)
    flat = flat[(flat["paper_norm"] != "") & (flat["metric_norm"] != "")]
    return flat.drop_duplicates()


def load_pwc_dataset(task_filter: str = "") -> pd.DataFrame:
    from datasets import load_dataset

    try:
        ds = load_dataset(
            "pwc-archive/evaluation-tables",
            split="train",
            token=_hf_dataset_token(),
        )
    except KeyError as e:
        if "maxdepth" in str(e).lower() or e.args == ("maxdepth",):
            raise RuntimeError(
                "Failed to load dataset from Hub (KeyError 'maxdepth'). "
                "Upgrade: pip install -U \"datasets>=3.0.0\" \"huggingface_hub>=0.26.0\" \"fsspec>=2024.10.0\" "
                "then restart the kernel."
            ) from e
        raise
    df = ds.to_pandas()
    if df.empty:
        raise RuntimeError("PwC dataset loaded but appears empty")

    # Nested schema (HF snapshot): task, datasets[], sota.rows[].metrics{}
    if "datasets" in df.columns and "task" in df.columns:
        return _pwc_flatten_nested_eval_tables(df, task_filter=task_filter)

    # Flat tabular schema (legacy snapshot)
    paper_col = first_present_column(df, PAPER_COL_CANDIDATES)
    metric_col = first_present_column(df, METRIC_COL_CANDIDATES)
    dataset_col = first_present_column(df, DATASET_COL_CANDIDATES)
    task_col = first_present_column(df, TASK_COL_CANDIDATES)
    if paper_col is None or metric_col is None:
        raise RuntimeError(
            "Unrecognized PwC schema. Columns: "
            f"{list(df.columns)}"
        )
    if task_filter and task_col is not None:
        tf = normalize_text(task_filter)
        keep = df[task_col].astype(str).map(normalize_text).str.contains(tf, na=False)
        df = df[keep].copy()
    if dataset_col is None:
        df["__dataset_fallback__"] = ""
        dataset_col = "__dataset_fallback__"
    out = pd.DataFrame({
        "paper_raw": df[paper_col],
        "dataset_raw": df[dataset_col],
        "metric_raw": df[metric_col],
    })
    out["paper_norm"] = out["paper_raw"].map(normalize_paper)
    out["dataset_norm"] = out["dataset_raw"].map(normalize_dataset)
    out["metric_norm"] = out["metric_raw"].map(normalize_metric)
    out = out[(out["paper_norm"] != "") & (out["metric_norm"] != "")]
    out["paper_match_key"] = out["paper_raw"].map(lambda pr: _pwc_identity_from_row(None, str(pr))[0])
    out["paper_match_kind"] = out["paper_raw"].map(lambda pr: _pwc_identity_from_row(None, str(pr))[1])
    return out.drop_duplicates()


In [ ]:
@dataclass
class PaperMatch:
    ours_paper_norm: str
    pwc_paper_norm: str
    score: float
    method: str


def match_papers(ours_papers: list[str], pwc_papers: list[str], threshold: float) -> list[PaperMatch]:
    pwc_set = set(pwc_papers)
    matches: list[PaperMatch] = []
    for p in sorted(set(ours_papers)):
        if p in pwc_set:
            matches.append(PaperMatch(p, p, 1.0, "exact"))
            continue
        best_name, best_score = "", 0.0
        for q in pwc_set:
            score = difflib.SequenceMatcher(None, p, q).ratio()
            if score > best_score:
                best_score, best_name = score, q
        if best_name and best_score >= threshold:
            matches.append(PaperMatch(p, best_name, best_score, "fuzzy"))
        else:
            matches.append(PaperMatch(p, "", best_score, "unmatched"))
    return matches

In [ ]:
def evaluate(ours_df: pd.DataFrame, pwc_df: pd.DataFrame, threshold: float) -> dict[str, pd.DataFrame]:
    ours_papers = sorted(ours_df["paper_norm"].unique().tolist())
    pwc_papers = sorted(pwc_df["paper_norm"].unique().tolist())
    paper_matches = match_papers(ours_papers, pwc_papers, threshold=threshold)
    paper_map = {m.ours_paper_norm: m.pwc_paper_norm for m in paper_matches if m.pwc_paper_norm}

    paper_matches_df = pd.DataFrame({
        "ours_paper_norm": [m.ours_paper_norm for m in paper_matches],
        "pwc_paper_norm": [m.pwc_paper_norm for m in paper_matches],
        "match_method": [m.method for m in paper_matches],
        "match_score": [round(float(m.score), 4) for m in paper_matches],
    }).sort_values(["match_method", "match_score", "ours_paper_norm"], ascending=[True, False, True])

    per_paper_rows = []
    tp_total = fp_total = fn_total = 0
    for ours_p in ours_papers:
        ours_metrics = set(ours_df.loc[ours_df["paper_norm"] == ours_p, "metric_norm"].tolist())
        pwc_p = paper_map.get(ours_p, "")
        pwc_metrics = set(pwc_df.loc[pwc_df["paper_norm"] == pwc_p, "metric_norm"].tolist()) if pwc_p else set()
        tp = len(ours_metrics & pwc_metrics)
        fp = len(ours_metrics - pwc_metrics)
        fn = len(pwc_metrics - ours_metrics)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = safe_f1(precision, recall)
        tp_total += tp
        fp_total += fp
        fn_total += fn
        per_paper_rows.append({
            "paper_norm": ours_p,
            "matched_pwc_paper_norm": pwc_p,
            "num_pred_metrics": len(ours_metrics),
            "num_gt_metrics": len(pwc_metrics),
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4),
            "pred_only_metrics": ", ".join(sorted(ours_metrics - pwc_metrics)),
            "gt_only_metrics": ", ".join(sorted(pwc_metrics - ours_metrics)),
        })

    per_paper_df = pd.DataFrame(per_paper_rows).sort_values(
        ["f1", "recall", "precision"], ascending=[True, True, True]
    )

    micro_p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = safe_f1(micro_p, micro_r)
    macro_p = float(per_paper_df["precision"].mean()) if len(per_paper_df) else 0.0
    macro_r = float(per_paper_df["recall"].mean()) if len(per_paper_df) else 0.0
    macro_f1 = float(per_paper_df["f1"].mean()) if len(per_paper_df) else 0.0

    matched_count = int((paper_matches_df["match_method"] != "unmatched").sum())
    unmatched_ours = sorted(set(ours_papers) - set(paper_map.keys()))
    unmatched_pwc = sorted(set(pwc_papers) - set(paper_map.values()))

    global_df = pd.DataFrame([
        {"metric": "micro_precision", "value": round(micro_p, 4)},
        {"metric": "micro_recall", "value": round(micro_r, 4)},
        {"metric": "micro_f1", "value": round(micro_f1, 4)},
        {"metric": "macro_precision", "value": round(macro_p, 4)},
        {"metric": "macro_recall", "value": round(macro_r, 4)},
        {"metric": "macro_f1", "value": round(macro_f1, 4)},
        {"metric": "papers_ours_total", "value": len(ours_papers)},
        {"metric": "papers_pwc_total", "value": len(pwc_papers)},
        {"metric": "papers_matched", "value": matched_count},
        {"metric": "papers_unmatched_ours", "value": len(unmatched_ours)},
        {"metric": "papers_unmatched_pwc", "value": len(unmatched_pwc)},
        {"metric": "tp_total", "value": tp_total},
        {"metric": "fp_total", "value": fp_total},
        {"metric": "fn_total", "value": fn_total},
    ])

    err_rows = []
    for _, row in per_paper_df.iterrows():
        for m in filter(None, [x.strip() for x in str(row["pred_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FP_metric", "metric": m})
        for m in filter(None, [x.strip() for x in str(row["gt_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FN_metric", "metric": m})

    return {
        "paper_matches": paper_matches_df,
        "per_paper_scores": per_paper_df,
        "global_scores": global_df,
        "errors": pd.DataFrame(err_rows),
        "unmatched_ours": pd.DataFrame({"paper_norm": unmatched_ours}),
        "unmatched_pwc": pd.DataFrame({"paper_norm": unmatched_pwc}),
    }


def enrich_ours_combinations_with_dataset_json(ours_df: pd.DataFrame, json_path: Path) -> pd.DataFrame:
    """Add `paper_match_key` to Combinations from dataset JSON arxiv/slug/paper_url (same logic as PwC)."""
    if not json_path.is_file():
        raise FileNotFoundError(json_path)
    with open(json_path, encoding="utf-8") as f:
        papers = json.load(f)
    title_map: dict[str, tuple[str, str]] = {}
    for o in papers:
        if not isinstance(o, dict):
            continue
        pn = normalize_paper(o.get("title"))
        if pn:
            title_map[pn] = _dataset_json_identity(o)
    out = ours_df.copy()
    keys, kinds = [], []
    for _, row in out.iterrows():
        pn = row["paper_norm"]
        if pn in title_map:
            keys.append(title_map[pn][0])
            kinds.append(title_map[pn][1])
        else:
            keys.append(f"title:{pn}")
            kinds.append("title")
    out["paper_match_key"] = keys
    out["paper_match_kind"] = kinds
    return out


def evaluate_combinations_id_aware(
    ours_df: pd.DataFrame,
    pwc_df: pd.DataFrame,
    threshold_title_fallback: float,
) -> dict[str, pd.DataFrame]:
    """Compare metric names grouped by `paper_match_key` (arxiv:/slug:/title:) instead of title-only fuzzy match."""
    if "paper_match_key" not in ours_df.columns or "paper_match_key" not in pwc_df.columns:
        raise ValueError("Expected column paper_match_key in both DataFrames.")

    def _agg_metric_set(s: pd.Series) -> set[str]:
        out: set[str] = set()
        for x in s.dropna().tolist():
            if str(x).strip():
                out.add(str(x))
        return out

    ours_g = ours_df.groupby("paper_match_key", sort=False)["metric_norm"].apply(_agg_metric_set)
    pwc_g = pwc_df.groupby("paper_match_key", sort=False)["metric_norm"].apply(_agg_metric_set)

    pwc_key_list = list(pwc_g.index)
    used_pwc: set[str] = set()
    key_map: dict[str, str] = {}
    for ok in ours_g.index:
        if ok in pwc_g.index:
            key_map[ok] = ok
            used_pwc.add(ok)
            continue
        if not str(ok).startswith("title:"):
            key_map[ok] = ""
            continue
        best_pk, best_sc = "", 0.0
        for pk in pwc_key_list:
            if pk in used_pwc:
                continue
            if not str(pk).startswith("title:"):
                continue
            sc = difflib.SequenceMatcher(None, ok, pk).ratio()
            if sc > best_sc:
                best_sc, best_pk = sc, pk
        if best_pk and best_sc >= threshold_title_fallback:
            key_map[ok] = best_pk
            used_pwc.add(best_pk)
        else:
            key_map[ok] = ""

    rep_norm = ours_df.groupby("paper_match_key", sort=False)["paper_norm"].first()

    per_paper_rows = []
    tp_total = fp_total = fn_total = 0
    for ok in ours_g.index:
        ours_m = set(ours_g[ok])
        pk = key_map.get(ok, "")
        pwc_m = set(pwc_g[pk]) if pk and pk in pwc_g.index else set()
        tp = len(ours_m & pwc_m)
        fp = len(ours_m - pwc_m)
        fn = len(pwc_m - ours_m)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = safe_f1(precision, recall)
        tp_total += tp
        fp_total += fp
        fn_total += fn
        if pk and pk in pwc_g.index:
            how = "exact_key" if ok == pk else "fuzzy_title_key"
        else:
            how = "unmatched"
        per_paper_rows.append(
            {
                "paper_norm": rep_norm.get(ok, ""),
                "ours_match_key": ok,
                "matched_pwc_match_key": pk,
                "match_kind": how,
                "num_pred_metrics": len(ours_m),
                "num_gt_metrics": len(pwc_m),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "pred_only_metrics": ", ".join(sorted(ours_m - pwc_m)),
                "gt_only_metrics": ", ".join(sorted(pwc_m - ours_m)),
            }
        )

    per_paper_df = pd.DataFrame(per_paper_rows).sort_values(
        ["f1", "recall", "precision"], ascending=[True, True, True]
    )

    micro_p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = safe_f1(micro_p, micro_r)
    macro_p = float(per_paper_df["precision"].mean()) if len(per_paper_df) else 0.0
    macro_r = float(per_paper_df["recall"].mean()) if len(per_paper_df) else 0.0
    macro_f1 = float(per_paper_df["f1"].mean()) if len(per_paper_df) else 0.0

    matched_count = int((per_paper_df["match_kind"] != "unmatched").sum())
    ours_keys = set(ours_g.index)
    pwc_keys = set(pwc_g.index)
    mapped_pwc = {key_map[k] for k in ours_keys if key_map.get(k)}
    unmatched_ours_keys = sorted(k for k in ours_keys if not key_map.get(k))

    paper_matches_df = pd.DataFrame(
        {
            "ours_paper_norm": per_paper_df["paper_norm"],
            "ours_match_key": per_paper_df["ours_match_key"],
            "pwc_match_key": per_paper_df["matched_pwc_match_key"],
            "match_method": per_paper_df["match_kind"],
            "match_score": per_paper_df["match_kind"].map(lambda x: 1.0 if x == "exact_key" else (0.9 if x == "fuzzy_title_key" else 0.0)),
        }
    )

    global_df = pd.DataFrame(
        [
            {"metric": "eval_mode", "value": "arxiv_slug_title_keys"},
            {"metric": "micro_precision", "value": round(micro_p, 4)},
            {"metric": "micro_recall", "value": round(micro_r, 4)},
            {"metric": "micro_f1", "value": round(micro_f1, 4)},
            {"metric": "macro_precision", "value": round(macro_p, 4)},
            {"metric": "macro_recall", "value": round(macro_r, 4)},
            {"metric": "macro_f1", "value": round(macro_f1, 4)},
            {"metric": "papers_ours_total", "value": len(ours_keys)},
            {"metric": "papers_pwc_total", "value": len(pwc_keys)},
            {"metric": "papers_matched", "value": matched_count},
            {"metric": "papers_unmatched_ours", "value": int(per_paper_df["match_kind"].eq("unmatched").sum())},
            {"metric": "papers_unmatched_pwc", "value": len(pwc_keys - mapped_pwc)},
            {"metric": "tp_total", "value": tp_total},
            {"metric": "fp_total", "value": fp_total},
            {"metric": "fn_total", "value": fn_total},
        ]
    )

    err_rows = []
    for _, row in per_paper_df.iterrows():
        for m in filter(None, [x.strip() for x in str(row["pred_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FP_metric", "metric": m})
        for m in filter(None, [x.strip() for x in str(row["gt_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FN_metric", "metric": m})

    return {
        "paper_matches": paper_matches_df,
        "per_paper_scores": per_paper_df,
        "global_scores": global_df,
        "errors": pd.DataFrame(err_rows),
        "unmatched_ours": pd.DataFrame({"paper_match_key": unmatched_ours_keys}),
        "unmatched_pwc": pd.DataFrame({"paper_match_key": sorted(pwc_keys - mapped_pwc)}),
    }


In [ ]:
repo_root = _find_repo_root_for_extraction()
ours_df = load_ours_combinations(INPUT_EXCEL, INPUT_SHEET)
pwc_df = load_pwc_dataset(task_filter=TASK_FILTER)
if EVAL_COMBINATIONS_USE_JSON_IDS:
    _jp_eval = (repo_root / "data" / DATASET_JSON_BASENAME).resolve()
    ours_eval = enrich_ours_combinations_with_dataset_json(ours_df, _jp_eval)
    results = evaluate_combinations_id_aware(ours_eval, pwc_df, threshold_title_fallback=MATCH_THRESHOLD)
    print("[eval Combinations vs PwC] id mode: arxiv/slug + paper_match_key on PwC (fuzzy only among title:…)")
else:
    results = evaluate(ours_df, pwc_df, threshold=MATCH_THRESHOLD)
    print("[eval Combinations vs PwC] normalized-title + fuzzy mode")

# Coverage: PDFs in corpus folder with no Combinations rows
pdf_dir_candidates = [
    repo_root / "data" / "pdf_files",
    repo_root / "table_extraction" / "kge_corpus",
    repo_root / "table_extraction" / "pdfs_prueba",
]
pdf_dir = next((p for p in pdf_dir_candidates if p.is_dir()), None)
if pdf_dir is None:
    raise FileNotFoundError("No PDF folder found for coverage_missing.")

pdf_stems = sorted({p.stem for p in pdf_dir.glob("*.pdf")})
pdf_norm_to_stem = {normalize_paper(s): s for s in pdf_stems if normalize_paper(s)}
ours_papers_norm = set(ours_df["paper_norm"].dropna().astype(str))

missing_norm = sorted(set(pdf_norm_to_stem.keys()) - ours_papers_norm)
coverage_missing_df = pd.DataFrame(
    {
        "missing_pdf_stem": [pdf_norm_to_stem[n] for n in missing_norm],
        "missing_pdf_norm": missing_norm,
    }
)

OUTPUT_EXCEL.parent.mkdir(parents=True, exist_ok=True)

papers_without_metrics_df = pd.DataFrame()
try:
    _audit = json_metrics_xml_vs_pwc_df
except NameError:
    _audit = pd.DataFrame()
if isinstance(_audit, pd.DataFrame) and not _audit.empty and "n_metrics_xml" in _audit.columns:
    _nm = pd.to_numeric(_audit["n_metrics_xml"], errors="coerce").fillna(0)
    papers_without_metrics_df = _audit.loc[_nm == 0].copy()
    _keep = [c for c in ["arxiv_id", "title", "repo_url", "mapping_state", "lightonocr_json_paths"] if c in papers_without_metrics_df.columns]
    if _keep:
        papers_without_metrics_df = papers_without_metrics_df[_keep]
else:
    _jp = (repo_root / "data" / DATASET_JSON_BASENAME).resolve()
    if _jp.is_file():
        with open(_jp, encoding="utf-8") as _f:
            _papers = json.load(_f)
        _rows = []
        for _o in _papers:
            if not isinstance(_o, dict):
                continue
            _m = _o.get("metrics")
            if isinstance(_m, list) and len(_m) == 0:
                _rows.append(
                    {
                        "arxiv_id": _o.get("arxiv_id"),
                        "title": _o.get("title"),
                        "repo_url": _o.get("repo_url"),
                        "local_path": _o.get("local_path"),
                    }
                )
        papers_without_metrics_df = pd.DataFrame(_rows)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    results["global_scores"].to_excel(writer, sheet_name="global_scores", index=False)
    coverage_missing_df.to_excel(writer, sheet_name="coverage_missing", index=False)
    papers_without_metrics_df.to_excel(writer, sheet_name="papers_without_metrics", index=False)

print(f"Report written: {OUTPUT_EXCEL.resolve()} (sheets: global_scores, coverage_missing, papers_without_metrics)")
print(f"Coverage missing (PDFs with no Combinations rows): {len(coverage_missing_df)}")
print(f"Papers with empty extracted metrics (metrics=[]): {len(papers_without_metrics_df)}")

In [ ]:
display(results["global_scores"])
display(results["per_paper_scores"].head(25))
display(results["paper_matches"].head(25))

In [ ]:
# Diagnostics: corpus PDFs vs Combinations papers
from pathlib import Path
import pandas as pd
import re
import unicodedata


def _strip_accents_local(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def _normalize_paper_local(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents_local(s)
    if s.startswith("http"):
        s = s.rstrip("/").split("/")[-1]
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


repo_root = _find_repo_root_for_extraction() if "_find_repo_root_for_extraction" in globals() else Path.cwd().resolve()

# 1) Resolve Combinations Excel path
if "INPUT_EXCEL" in globals() and Path(INPUT_EXCEL).exists():
    comb_xlsx = Path(INPUT_EXCEL)
else:
    mode = MODE if "MODE" in globals() else "filtered"
    candidates = [
        repo_root / "table_extraction" / "kge_corpus" / f"gliner2_lightonocr_combinations_{mode}.xlsx",
        repo_root / "table_extraction" / "kge_corpus" / "gliner2_lightonocr_combinations_filtered.xlsx",
        repo_root / "table_extraction" / "kge_corpus" / "gliner2_lightonocr_combinations_raw.xlsx",
        repo_root / "data" / "pdf_files" / f"gliner2_lightonocr_combinations_{mode}.xlsx",
        repo_root / "table_extraction" / "pdfs_prueba" / f"gliner2_lightonocr_combinations_{mode}.xlsx",
    ]
    comb_xlsx = next((p for p in candidates if p.exists()), None)

if comb_xlsx is None:
    raise FileNotFoundError("Combinations Excel not found for coverage diagnostic.")

# 2) Resolve PDF source folder (priority: data/pdf_files)
pdf_dir_candidates = [
    repo_root / "data" / "pdf_files",
    repo_root / "table_extraction" / "kge_corpus",
    repo_root / "table_extraction" / "pdfs_prueba",
]
pdf_dir = next((p for p in pdf_dir_candidates if p.is_dir()), None)
if pdf_dir is None:
    raise FileNotFoundError("No PDF folder found for coverage comparison.")

# 3) Load Combinations sheet
comb = pd.read_excel(comb_xlsx, sheet_name="Combinations")
if "paper" not in comb.columns:
    raise ValueError(f"Combinations sheet has no 'paper' column. Columns: {list(comb.columns)}")

pdf_stems_raw = sorted({p.stem for p in pdf_dir.glob("*.pdf")})
pdf_norm = {_normalize_paper_local(s): s for s in pdf_stems_raw if _normalize_paper_local(s)}

papers_raw = comb["paper"].dropna().astype(str).tolist()
papers_norm_set = {_normalize_paper_local(x) for x in papers_raw if _normalize_paper_local(x)}

missing_norm = sorted(set(pdf_norm.keys()) - papers_norm_set)
present_norm = sorted(set(pdf_norm.keys()) & papers_norm_set)

missing_df = pd.DataFrame({
    "missing_pdf_stem": [pdf_norm[n] for n in missing_norm],
    "missing_pdf_norm": missing_norm,
})

present_df = pd.DataFrame({
    "present_pdf_stem": [pdf_norm[n] for n in present_norm],
    "present_pdf_norm": present_norm,
})

print("Excel Combinations:", comb_xlsx)
print("PDF folder used:", pdf_dir)
print("PDFs totales:", len(pdf_stems_raw))
print("Unique papers in Combinations (normalized):", len(papers_norm_set))
print("PDFs present in Combinations:", len(present_norm))
print("PDFs missing from Combinations:", len(missing_norm))

print("\n=== First missing (up to 50) ===")
display(missing_df.head(50))

print("\n=== Sample of present (up to 20) ===")
display(present_df.head(20))

print("\nNote: this check is also written to the 'coverage_missing' sheet in the main Excel report.")

In [ ]:
# Diagnostics: Hugging Face PwC dataset sample
from datasets import load_dataset
import pandas as pd
import os

HF_DATASET_ID = "pwc-archive/evaluation-tables"
HF_SPLIT = "train"

# Do not hardcode tokens in the notebook
# Use HF_TOKEN in the environment or huggingface-cli / notebook_login beforehand
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
hf_token = hf_token.strip() if hf_token and hf_token.strip() else None

print(f"Loading dataset: {HF_DATASET_ID} [{HF_SPLIT}] ...")
ds_raw = load_dataset(HF_DATASET_ID, split=HF_SPLIT, token=hf_token)
df_raw = ds_raw.to_pandas()

print("\n=== Overview ===")
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
print("Column names:")
for c in df_raw.columns:
    print(" -", c)

print("\n=== dtypes ===")
print(df_raw.dtypes.to_string())

print("\n=== Non-null counts per column ===")
non_null = df_raw.notna().sum().sort_values(ascending=False)
print(non_null.to_string())

print("\n=== Sample (3 rows, key columns if present) ===")
key_cols = [c for c in ["task", "description", "source_link", "datasets", "subtasks", "categories", "synonyms"] if c in df_raw.columns]
if key_cols:
    display(df_raw[key_cols].head(3))
else:
    display(df_raw.head(3))

if "task" in df_raw.columns:
    print("\n=== Top tasks (task) ===")
    print(df_raw["task"].astype(str).value_counts().head(15).to_string())

if "datasets" in df_raw.columns:
    def _as_list(x):
        if x is None:
            return []
        if hasattr(x, "tolist"):
            try:
                x = x.tolist()
            except Exception:
                return []
        if isinstance(x, (list, tuple)):
            return list(x)
        return [x]

    def _count_nodes_and_rows(dataset_nodes):
        nodes = 0
        rows = 0
        stack = [d for d in _as_list(dataset_nodes) if isinstance(d, dict)]
        while stack:
            node = stack.pop()
            nodes += 1
            sota = node.get("sota") if isinstance(node.get("sota"), dict) else None
            if sota:
                mrows = _as_list(sota.get("rows"))
                rows += sum(1 for r in mrows if isinstance(r, dict))
            subs = _as_list(node.get("subdatasets"))
            stack.extend(sd for sd in subs if isinstance(sd, dict))
        return nodes, rows

    stats = df_raw["datasets"].apply(_count_nodes_and_rows)
    nodes_per_task = stats.map(lambda t: t[0])
    rows_per_task = stats.map(lambda t: t[1])

    print("\n=== Nested `datasets` field stats ===")
    print("Total dataset-nodes:", int(nodes_per_task.sum()))
    print("Total sota.rows:", int(rows_per_task.sum()))
    print("Mean dataset-nodes per task:", round(float(nodes_per_task.mean()), 2))
    print("Mean sota.rows per task:", round(float(rows_per_task.mean()), 2))

    df_stats = pd.DataFrame({
        "task": df_raw["task"] if "task" in df_raw.columns else pd.Series(range(len(df_raw))),
        "dataset_nodes": nodes_per_task,
        "sota_rows": rows_per_task,
    }).sort_values("sota_rows", ascending=False)

    print("\nTop 15 tasks by sota.rows count:")
    display(df_stats.head(15))

print("\nDataset diagnostic finished.")